# Dafne Thigh — Asian Dataset Evaluation (Thigh, Water)

Computes per-muscle Dice / Hausdorff metrics for Dafne Thigh segmentations on the **MRI_data_asian** thigh water images.

- **Predictions**: `asian_segs_water/{subject}/Thigh/Water_dafne_thigh.npz`
- **Ground truth**: `MRI_data_asian/MRI_data/{subject}/Thigh/mask_muscles.nii.gz` (labels 1–13)

**The Asian GT is unilateral** — each label covers only one side; the other side is background.  
`_L` and `_R` NPZ keys are therefore evaluated **separately** against the same GT label.  
Run `dafne_asian_water_lambda.ipynb` first to generate the NPZ files.

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [ ]:
BOUNDARY_DISTANCE = 1

SEG_DIR    = os.path.join('..', 'asian_segs_water')
DATA_ROOT  = os.path.join('..', '..', 'MRI_data_asian', 'MRI_data')
RESULT_DIR = 'results_asian_water'
os.makedirs(RESULT_DIR, exist_ok=True)

# (muscle_name, asian_gt_label, dafne_npz_keys)
# GT is UNILATERAL — L and R NPZ keys evaluated separately against the same GT label.
MUSCLES = [
    ('sartorius_L', 5, ['Sartorius_L']),
    ('sartorius_R', 5, ['Sartorius_R']),
    ('gracilis_L',  6, ['Gracilis_L' ]),
    ('gracilis_R',  6, ['Gracilis_R' ]),
]

seg_files = sorted(glob.glob(os.path.join(SEG_DIR, '*', 'Thigh', 'Water_dafne_thigh.npz')))
print(f'SEG_DIR   : {os.path.abspath(SEG_DIR)}')
print(f'DATA_ROOT : {os.path.abspath(DATA_ROOT)}')
print(f'RESULT_DIR: {os.path.abspath(RESULT_DIR)}')
print(f'Found     : {len(seg_files)} NPZ files')

In [ ]:
# ── Sanity check: show what keys are actually in the NPZ files ────────────────
if seg_files:
    sample = np.load(seg_files[0])
    print(f'Keys in {os.path.basename(seg_files[0])}: {sorted(sample.files)}')

In [ ]:
def evaluate_muscle(muscle_name, gt_label, dafne_keys, seg_files, result_dir):
    """
    dafne_keys: list of NPZ key strings to OR together into one prediction mask.
    """
    results = []
    for seg_path in seg_files:
        parts   = seg_path.replace('\\', '/').split('/')
        subject = parts[-3]

        gt_path = os.path.join(DATA_ROOT, subject, 'Thigh', 'mask_muscles.nii.gz')
        if not os.path.exists(gt_path):
            print(f'  GT not found: {gt_path}, skipping')
            continue

        gt_image = sitk.ReadImage(gt_path)
        gt       = sitk.Cast(gt_image == gt_label, sitk.sitkUInt8)
        gt_arr   = sitk.GetArrayFromImage(gt).astype(float)

        npz_data = np.load(seg_path)
        pred_arr = np.zeros(gt_arr.shape, dtype=np.uint8)
        for key in dafne_keys:
            if key in npz_data:
                pred_arr |= npz_data[key].astype(np.uint8)
            else:
                print(f'  [{subject}] key "{key}" not found in NPZ — treating as zero')

        pred_sitk = sitk.GetImageFromArray(pred_arr)
        pred_sitk.CopyInformation(gt_image)
        pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

        dice_filter = sitk.LabelOverlapMeasuresImageFilter()
        dice_filter.Execute(gt, pred)

        if gt_arr.sum() > 0 and pred_arr.sum() > 0:
            hd_filter = sitk.HausdorffDistanceImageFilter()
            hd_filter.Execute(gt, pred)
            hd = hd_filter.GetHausdorffDistance()
        else:
            hd = np.nan

        results.append({
            'subject':                              subject,
            'seg_file':                             os.path.basename(seg_path),
            f'{muscle_name}_dice':                  dice_filter.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_filter.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_filter.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_filter.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_filter.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
        })

    df       = pd.DataFrame(results)
    csv_path = os.path.join(result_dir, f'df_{muscle_name}_dafne_asian_water.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows → {csv_path}')
    return df

In [ ]:
# ── Run evaluation ────────────────────────────────────────────────────────────

dfs = {}
for muscle_name, gt_label, dafne_keys in MUSCLES:
    print(f'\n── {muscle_name}  (GT={gt_label}, keys={dafne_keys}) ──')
    dfs[muscle_name] = evaluate_muscle(
        muscle_name, gt_label, dafne_keys, seg_files, RESULT_DIR
    )

print('\nDone.')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────

from IPython.display import display

summary_rows = []
for muscle_name, df in dfs.items():
    dice_col = f'{muscle_name}_dice'
    hd_col   = f'{muscle_name}_hausdorff'
    summary_rows.append({
        'muscle':         muscle_name,
        'n':              len(df),
        'dice_mean':      df[dice_col].mean(),
        'dice_std':       df[dice_col].std(),
        'hausdorff_mean': df[hd_col].mean(),
        'hausdorff_std':  df[hd_col].std(),
    })

summary = pd.DataFrame(summary_rows).set_index('muscle')
summary_path = os.path.join(RESULT_DIR, 'summary_dafne_asian_water.csv')
summary.to_csv(summary_path)
print(f'Summary saved → {summary_path}\n')
display(summary.round(4))